# Lab 5 — Dynamic INT8 quantization
Apply dynamic weight quantization to the fully-connected classifier. Browser deployment uses verified FP32 by default; INT8 browser execution is an optional compatibility experiment.


In [ ]:
!pip -q install onnx onnxruntime numpy pandas matplotlib


In [ ]:
from google.colab import files
uploaded=files.upload() # select gesture_model.onnx; optionally select a session CSV
import os, numpy as np, onnx, onnxruntime as ort, time, pandas as pd
model_path='gesture_model.onnx'
assert os.path.exists(model_path), 'Please upload gesture_model.onnx'
onnx.checker.check_model(onnx.load(model_path))
options=ort.SessionOptions();options.intra_op_num_threads=1
sess=ort.InferenceSession(model_path,options,providers=['CPUExecutionProvider'])
features=np.zeros((1,42),dtype=np.float32)
print('Input',sess.get_inputs()[0].name,sess.get_inputs()[0].shape)
print('Logits',sess.run(None,{sess.get_inputs()[0].name:features})[0])


In [ ]:
from onnxruntime.quantization import quantize_dynamic,QuantType
quantize_dynamic('gesture_model.onnx','gesture_model_int8.onnx',weight_type=QuantType.QInt8)
onnx.checker.check_model(onnx.load('gesture_model_int8.onnx'))
q=ort.InferenceSession('gesture_model_int8.onnx',options,providers=['CPUExecutionProvider'])
print('FP32 bytes',os.path.getsize('gesture_model.onnx'))
print('INT8 bytes',os.path.getsize('gesture_model_int8.onnx'))
# TODO: compare model size ratio and explain metadata overhead.


In [ ]:
uploaded=files.upload() # upload a held-out test-session CSV
csvfile=next(n for n in uploaded if n.endswith('.csv'))
df=pd.read_csv(csvfile)
classes=['open_palm','fist','peace','thumbs_up']
x=df[[f'f{i}' for i in range(42)]].to_numpy(dtype=np.float32)
y=np.array([classes.index(v) for v in df.label],dtype=np.int64)
p=sess.run(None,{sess.get_inputs()[0].name:x})[0]
qi=q.run(None,{q.get_inputs()[0].name:x})[0]
print('FP32 accuracy',np.mean(p.argmax(1)==y));print('INT8 accuracy',np.mean(qi.argmax(1)==y))
print('Prediction agreement',np.mean(p.argmax(1)==qi.argmax(1)))


In [ ]:
# FP32 and INT8 use the same thread count. Modification: rerun the notebook for Tiny, Small and Large,
# setting MODEL_NAME to the capacity you uploaded; combine the CSVs into your per-model comparison table.
MODEL_NAME='Small'
THREADS=1
BATCHES=[1,32]
options=ort.SessionOptions();options.intra_op_num_threads=THREADS
variants={'FP32':'gesture_model.onnx','INT8 dynamic':'gesture_model_int8.onnx'}
def ms(session,x,repeat=200):
  feed={session.get_inputs()[0].name:x};[session.run(None,feed) for _ in range(20)]
  times=[]
  for _ in range(repeat):
    t=time.perf_counter();session.run(None,feed);times.append((time.perf_counter()-t)*1000)
  return np.median(times),np.percentile(times,95)
rows=[]
for variant,model_file in variants.items():
  s=ort.InferenceSession(model_file,options,providers=['CPUExecutionProvider'])
  pred=s.run(None,{s.get_inputs()[0].name:x})[0].argmax(1)
  for batch in BATCHES:
    arr=x[:batch] if len(x)>=batch else np.resize(x,(batch,42))
    med,p95=ms(s,arr)
    rows.append({'model':MODEL_NAME,'variant':variant,'file_bytes':os.path.getsize(model_file),
                 'test_accuracy':np.mean(pred==y),'agreement_with_fp32':np.mean(pred==p.argmax(1)),
                 'threads':THREADS,'batch':batch,'median_ms':med,'p95_ms':p95})
results=pd.DataFrame(rows)
print(results.round(4).to_string(index=False))
csv_name=f'lab5_{MODEL_NAME}_threads{THREADS}.csv'
results.to_csv(csv_name,index=False);files.download(csv_name)
files.download('gesture_model_int8.onnx')